# Adaptive Pole-Stability Reward Shaping: Final Test (DQN)

**Assignment 3 strategy:** the training reward was modified so that the agent receives a larger reward when the pole is close to upright. This notebook is the **final test** of the best validation checkpoint (`weights/gaussian_pole_stability.pth`) produced by `adaptive_pole_stability_reward_shaping.ipynb`. The provided DQN architecture is unchanged.

## Research question

Does a smooth Gaussian reward based on pole angle improve DQN performance across pole lengths from 0.4 to 1.8, including the disturbance and increasing-force phases used by the supplied evaluation script?

## Hypothesis

H1: A dense, angle-sensitive reward will encourage the agent to keep the pole closer to upright, and will improve robustness across pole lengths and disturbances.
H01: A dense, angle-sensitive reward will not encourage the agent to keep the pole closer to upright.

H2: A dense, angle-sensitive reward will significantly improve robustness across pole lengths and disturbances.
H02: A dense, angle-sensitive reward will not significantly improve robustness across pole lengths and disturbances.

## How the hypotheses are tested here

- **H1 (upright behaviour):** besides the episode score, every test run records the **mean absolute pole angle** (radians). A lower value means the agent kept the pole closer to upright.
- **H2 (robustness):** the **score** (standard environment reward) over 30 pole lengths x 10 runs under the supplied force schedule, plus the fraction of runs that survive the disturbance phase (> 500) and the increasing-force phase (> 1000).
- The exact Gaussian shaped reward from training is also computed on the test episodes, **for reporting only**. It never changes the dynamics or the score.

## Important test-design detail

The supplied evaluator ignores `truncated`, so episodes run past 500 steps. Between scores of 500 and 1000 the force is set to 75 whenever the score is a multiple of 25 (disturbance), and above 1000 it grows as `25 + 0.01 * score`. The score is the **standard CartPole reward**, not the Gaussian reward, so the two metrics must be reported separately.

## 1. Imports and reproducibility

Set `SIGMA` and `TERMINAL_PENALTY` to the **same values used in training** (they are imported from `network` when available, as in the training notebook).

In [1]:
import random
from collections import deque
from dataclasses import dataclass
from pathlib import Path
from scipy import stats
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import gymnasium as gym
import time 
import matplotlib.pyplot as plt
import random
from network import *

STABILITY_THRESHOLD = 40
STABILITY_BONUS = 1.0

# Supplied evaluation setup: 30 pole lengths from 0.4 to 1.8, 10 runs per length.
TEST_LENGTHS = np.linspace(0.4, 1.8, 30)
RUNS_PER_LENGTH = 10
WIND = 25
SEED = 42

WEIGHTS_PATH = Path("weights") / "gaussian_pole_stability.pth"
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cpu")

ModuleNotFoundError: No module named 'network'

In [ ]:
def gaussian_stability_reward(pole_angle, sigma=SIGMA):
    """Return a smooth reward in [0, 1], maximized at pole_angle == 0."""
    if sigma <= 0:
        raise ValueError("sigma must be positive")
    return float(np.exp(-(float(pole_angle) ** 2) / (2.0 * sigma ** 2)))


def shaped_reward(observation, terminated, steps, sigma=SIGMA,
                  terminal_penalty=TERMINAL_PENALTY,
                  stability_threshold=STABILITY_THRESHOLD,
                  stability_bonus=STABILITY_BONUS):
    """Reward for one transition. `observation` is the next observation."""
    angle = float(observation[2])
    reward = gaussian_stability_reward(angle, sigma=sigma)
    if terminated:
        reward -= terminal_penalty
    if steps == stability_threshold and not terminated:
        reward += stability_bonus
        
    return reward

## 3. Test episode (supplied force schedule)

`run_test_episode()` follows the supplied `test_pole_length()` step by step: greedy action, `done` taken from `terminated` only, score = accumulated **standard** reward, and the same wind / increasing-force rules.

Additions that do **not** affect the dynamics or the score:

- an optional `seed` for reproducible resets,
- the Gaussian shaped return of the episode (reported separately),
- the mean absolute pole angle (evidence for H1).

In [ ]:
def run_test_episode(env, q_network, seed=None):
    wind = WIND
    state = env.reset(seed=seed)[0]
    state = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
    done = False
    total_reward = 0
    steps = 0
    shaped_return = 0.0
    abs_angles = []

    while not done:
        with torch.no_grad():
            action = q_network(state).argmax().item()
        next_state, reward, done, _, __ = env.step(action)

        # Reporting only: Gaussian reward as used in training.
        steps += 1
        shaped_return += shaped_reward(next_state, done, steps)
        abs_angles.append(abs(float(next_state[2])))

        next_state = torch.tensor(next_state, dtype=torch.float32).unsqueeze(0)
        state = next_state
        total_reward += reward

        # Supplied force schedule (unchanged).
        if total_reward >= 500 and total_reward <= 1000:
            if total_reward % wind == 0:
                env.unwrapped.force_mag = 75

        if total_reward > 1000:
            env.unwrapped.force_mag = 25 + (0.01 * total_reward)

    return {
        "score": float(total_reward),
        "shaped_return": float(shaped_return),
        "mean_abs_angle": float(np.mean(abs_angles)),
    }

## 4. Load the best validation checkpoint

The checkpoint selected on the validation lengths is loaded once. The final test is used only for the final reported result; do not tune the model against it.

In [ ]:
if not WEIGHTS_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {WEIGHTS_PATH}. Run adaptive_pole_stability_reward_shaping.ipynb "
        "first and make sure the weights/ directory is in this project directory."
    )

_env = gym.make("CartPole-v1")
state_dim = _env.observation_space.shape[0]
action_dim = _env.action_space.n
_env.close()

q_network = QNetwork(state_dim, action_dim).to(device)
q_network.load_state_dict(torch.load(WEIGHTS_PATH, map_location=device, weights_only=True))
q_network.eval()
print("Loaded:", WEIGHTS_PATH)

## 5. Final test over 30 pole lengths

Each run uses a fresh environment with `env.unwrapped.length` set to the test length (a fresh environment also resets `force_mag`). Every episode is stored, so mean, standard deviation and any statistical test can be recomputed from the CSV files.

> Episodes can become long because the agent keeps balancing past 500 steps; this is expected under the supplied evaluator. The increasing force eventually ends the episode.

In [ ]:
def final_test(q_network, lengths=TEST_LENGTHS, runs_per_length=RUNS_PER_LENGTH):
    rows = []
    for length_idx, length in enumerate(lengths):
        print(f"Testing pole length: {length:.3f}")
        for run_idx in range(runs_per_length):
            env = gym.make("CartPole-v1")
            env.unwrapped.length = float(length)
            result = run_test_episode(
                env, q_network, seed=SEED + 1000 * length_idx + run_idx
            )
            env.close()
            rows.append({
                "pole_length": round(float(length), 2),
                "run": run_idx + 1,
                **result,
            })
    return pd.DataFrame(rows)


test_runs = final_test(q_network)
test_runs.to_csv(RESULTS_DIR / "test_runs_reward_shaping.csv", index=False)

by_length = (
    test_runs.groupby("pole_length")
    .agg(
        mean_score=("score", "mean"),
        std_score=("score", lambda x: float(np.std(x))),
        mean_shaped_return=("shaped_return", "mean"),
        mean_abs_angle=("mean_abs_angle", "mean"),
        survive_over_500=("score", lambda x: float(np.mean(x > 500))),
        survive_over_1000=("score", lambda x: float(np.mean(x > 1000))),
    )
    .reset_index()
)
by_length.to_csv(RESULTS_DIR / "test_by_length_reward_shaping.csv", index=False)

overall_avg = float(by_length["mean_score"].mean())
total_score = float(by_length["mean_score"].sum())
print(f"\nOverall average score over {len(by_length)} pole lengths: {overall_avg:.1f}")
print(f"Total (sum of per-length means): {total_score:.1f}")
print("Saved results/test_runs_reward_shaping.csv, results/test_by_length_reward_shaping.csv")

## 6. Test plots

The first plot reproduces the supplied bar chart (mean score per pole length, error bars = standard deviation over 10 runs). The others show the evidence for H1 (pole angle) and the shaped return for reference.

In [ ]:
x = np.arange(len(by_length))
labels = [f"{l:.2f}" for l in by_length["pole_length"]]

plt.figure(figsize=(10, 5))
plt.bar(x, by_length["mean_score"], yerr=by_length["std_score"], capsize=4, alpha=0.7)
plt.xticks(x, labels, rotation=45)
plt.xlabel("Pole length")
plt.ylabel("Episode length")
plt.title(f"Adaptive reward shaping: average score over pole lengths = {overall_avg:.1f}")
plt.tight_layout()
plt.savefig("adaptive_pole_stability_test_results.png", dpi=150)
plt.show()

plt.figure(figsize=(9, 4))
plt.plot(by_length["pole_length"], by_length["mean_abs_angle"], marker="o")
plt.xlabel("Pole length")
plt.ylabel("Mean absolute pole angle (radians)")
plt.title("Upright behaviour during the final test (lower = closer to upright)")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 4))
plt.plot(by_length["pole_length"], by_length["survive_over_500"], marker="o", label="score > 500 (disturbance phase)")
plt.plot(by_length["pole_length"], by_length["survive_over_1000"], marker="s", label="score > 1000 (increasing force)")
plt.xlabel("Pole length")
plt.ylabel("Fraction of runs")
plt.title("Robustness: fraction of runs surviving each phase")
plt.ylim(-0.05, 1.05)
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 4))
plt.plot(by_length["pole_length"], by_length["mean_shaped_return"], marker="o")
plt.xlabel("Pole length")
plt.ylabel("Mean Gaussian shaped return")
plt.title("Shaped return on test episodes (reference only, not the test score)")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

display(by_length)

## 7. Save the supplied-format results

Same wide format as the supplied evaluator (`Avg_<length>`, `Std_<length>`, `Total`), so the file can be compared directly with the other groups' results.

In [ ]:
"""results = {}
for _, row in by_length.iterrows():
    results[f"Avg_{row['pole_length']}"] = row["mean_score"]
    results[f"Std_{row['pole_length']}"] = row["std_score"]
results["Total"] = total_score

pd.DataFrame(results).to_csv('results/test_results_reward_shaping.csv', index=False)"""

## 8. Comparison with the baseline DQN (optional)

If the group's baseline DQN (standard CartPole reward) was tested with this same notebook structure, save its per-run file as `results/test_runs_baseline.csv` (same columns as `results/test_runs_reward_shaping.csv`). The cell below then compares both across the 30 lengths using the per-length mean over runs as the paired unit.

- **H1:** compares mean absolute pole angle (lower for the shaped agent supports H1).
- **H2:** compares score (higher for the shaped agent supports H2).

A significant difference needs a statistical test; with one trained agent per method, the result describes that training run, not training randomness. Repeat training with several seeds if your analysis plan requires it.

In [ ]:
BASELINE_RUNS = RESULTS_DIR / "test_runs_baseline.csv"

if BASELINE_RUNS.exists():
    base_runs = pd.read_csv(BASELINE_RUNS)
    base = base_runs.groupby("pole_length").agg(
        score=("score", "mean"), angle=("mean_abs_angle", "mean")
    )
    shaped = test_runs.groupby("pole_length").agg(
        score=("score", "mean"), angle=("mean_abs_angle", "mean")
    )
    common = shaped.index.intersection(base.index)

    summary = pd.DataFrame({
        "metric": ["Overall mean score", "Overall mean |angle| (rad)"],
        "baseline": [base.loc[common, "score"].mean(), base.loc[common, "angle"].mean()],
        "reward_shaping": [shaped.loc[common, "score"].mean(), shaped.loc[common, "angle"].mean()],
    })
    display(summary)

    for name, col in [("score (H2)", "score"), ("mean |angle| (H1)", "angle")]:
        t_stat, p_val = stats.ttest_rel(shaped.loc[common, col], base.loc[common, col])
        w_stat, w_p = stats.wilcoxon(shaped.loc[common, col], base.loc[common, col])
        print(f"{name}: paired t-test t={t_stat:.3f}, p={p_val:.4f} | Wilcoxon p={w_p:.4f}")
else:
    print(f"{BASELINE_RUNS} not found. Skipping baseline comparison.")

## 9. Evaluation and reporting checklist

Parameters to record:
- Mean and standard deviation of score for each of the 30 pole lengths (`results/test_by_length_reward_shaping.csv`).
- Overall average across all 30 lengths (printed in section 5 and in the bar-chart title).
- Mean absolute pole angle and the fraction of runs surviving the disturbance (> 500) and increasing-force (> 1000) phases, as evidence for H1 and H2.
- The same metrics for the group's baseline DQN, trained with the standard CartPole reward.
- The exact training-length schedule, number of episodes, seed(s), sigma, terminal penalty, and all unchanged DQN hyperparameters (from the training notebook).
- A statistical comparison across repeated runs if required by your group's analysis plan.
- The tested checkpoint is `weights/gaussian_pole_stability.pth` (best validation checkpoint).
- The final test accumulates the environment's standard reward and applies the supplied force schedule. It does not use the Gaussian reward, so report the shaped return and the test score separately.
- Use validation for checkpoint selection and the final test for the final reported result. Avoid re-running training until the test score improves.

*Use the **provided evaluation script's score/episode performance** for the main comparison.

## 10. References

1. Ng, A. Y., Harada, D., & Russell, S. (1999). *Policy Invariance Under Reward Transformations: Theory and Application to Reward Shaping*. Proceedings of ICML.
   https://ai.stanford.edu/~ang/papers/shaping-icml99.pdf

2. Suay, H. B., et al. (2015). *Reward Shaping by Demonstration*. Verify the publication metadata and the specific claims you attribute to this work before submission.
   https://publications.ri.cmu.edu/reward-shaping-by-demonstration